# Hybrid Classical-Quantum CCTV Anomaly Detection (UCF-Crime)

**Dataset:** UCF-Crime. Set `DATA_ROOT` to your dataset path (see `DATASET_SETUP.md`).

## 1. Config and paths

In [ ]:
import os
import sys
from pathlib import Path
import torch

# ---------------- CONFIG ---------------- #

DATA_ROOT = "/kaggle/input/ucf-crime-dataset"

# Temporal settings
FRAMES_PER_VIDEO = 16        # 16-frame clip (good for LSTM)
SAMPLE_FPS = 10               # Take 1 frame every 3 frames (~10 FPS if video is 30 FPS)

# Image settings
IMG_SIZE = 224               # Required for ResNet50

# Training settings
BATCH_SIZE = 4               # Safe for ResNet + LSTM + Quantum
EPOCHS = 30                  # Now safe to train up to 10
LR = 1e-4

# Quantum settings
N_QUBITS = 4
REDUCED_DIM = 4            

# Dataset balancing
MAX_NORMAL_VIDEOS = 50       # Limit only NormalVideos
MAX_CLIPS_PER_CLASS = 1500    # Balance clips per class
MAX_NORMAL_CLIPS=1000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Class labels
CLASSES = [
    "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",
    "Fighting", "NormalVideos", "RoadAccidents", "Robbery", "Shooting",
    "Shoplifting", "Stealing", "Vandalism"
]

NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

print("DATA_ROOT:", DATA_ROOT)
print("DEVICE:", DEVICE)
print("NUM_CLASSES:", NUM_CLASSES)


DATA_ROOT: /kaggle/input/ucf-crime-dataset
DEVICE: cuda
NUM_CLASSES: 14


## 2. Dependencies (Kaggle: skip or run only if missing)

In [ ]:
# Uncomment and run if PennyLane is not installed (Kaggle may have it)
!pip install pennylane torch torchvision opencv-python-headless

## 3. List videos by class

In [ ]:
from collections import defaultdict
import random

def get_frame_index(fname):
    return int(fname.split("_")[-1].split(".")[0])

def get_video_id(fname):
    return fname.rsplit("_", 1)[0]

def build_clip_list(data_root, split):
    root = Path(data_root) / split
    clips = []

    print(f"\nBuilding {split} clips...")

    for class_name in CLASSES:
        class_dir = root / class_name
        if not class_dir.exists():
            continue

        by_video = defaultdict(list)

        # -------- Special handling for NormalVideos -------- #
        if split == "Train" and class_name == "NormalVideos":
            print("Processing NormalVideos with clip limit...")

            for f in class_dir.iterdir():
                if not f.is_file() or f.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
                    continue

                vid = get_video_id(f.name)

                if len(by_video) >= MAX_NORMAL_VIDEOS and vid not in by_video:
                    continue

                by_video[vid].append((get_frame_index(f.name), str(f)))

        else:
            for f in class_dir.iterdir():
                if f.is_file() and f.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                    vid = get_video_id(f.name)
                    by_video[vid].append((get_frame_index(f.name), str(f)))

        class_clips = []

        # -------- Create clips -------- #
        for vid, frames in by_video.items():
            frames = sorted(frames, key=lambda x: x[0])
            paths = [p for _, p in frames]

            for start in range(0, len(paths) - FRAMES_PER_VIDEO + 1, FRAMES_PER_VIDEO):
                clip_paths = paths[start:start + FRAMES_PER_VIDEO]
                class_clips.append((clip_paths, class_to_idx[class_name]))

        # -------- Clip Limiting Logic -------- #
        if split == "Train":
            if class_name == "NormalVideos":
                if len(class_clips) > MAX_NORMAL_CLIPS:
                    class_clips = random.sample(class_clips, MAX_NORMAL_CLIPS)
            else:
                if len(class_clips) > MAX_CLIPS_PER_CLASS:
                    class_clips = random.sample(class_clips, MAX_CLIPS_PER_CLASS)

        print(f"{split} / {class_name}: {len(class_clips)} clips")

        clips.extend(class_clips)

    random.shuffle(clips)
    return clips


train_clips = build_clip_list(DATA_ROOT, "Train")
test_clips = build_clip_list(DATA_ROOT, "Test")

print("Train clips:", len(train_clips))
print("Test clips:", len(test_clips))


## 4. Frame extraction and Dataset

In [ ]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

def load_clip(paths, size=224, debug_fail=False):
    """Load list of image paths -> (T, C, H, W) tensor, ImageNet normalized."""
    frames = []
    for p in paths:
        im = cv2.imread(p)
        if im is None:
            if debug_fail:
                print(f"  [load_clip] Failed to read: {p}")
            continue
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        im = cv2.resize(im, (size, size))
        frames.append(im)
    if len(frames) < len(paths):
        return None
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    arr = (arr - mean) / std
    return torch.from_numpy(arr).permute(0, 3, 1, 2)  # (T, C, H, W)

class UCFCrimeClipDataset(Dataset):
    def __init__(self, clip_list, size=224):
        self.clip_list = clip_list
        self.size = size

    def __len__(self):
        return len(self.clip_list)

    def __getitem__(self, i):
        paths, label = self.clip_list[i]
        x = load_clip(paths, self.size)
        if x is None:
            return self.__getitem__((i + 1) % len(self.clip_list))
        return x, label

train_ds = UCFCrimeClipDataset(train_clips, size=IMG_SIZE)
test_ds = UCFCrimeClipDataset(test_clips, size=IMG_SIZE)
print("Dataset sizes: train =", len(train_ds), ", test =", len(test_ds))

# Load one sample to verify shape
sample_x, sample_y = train_ds[0]
print("Sample clip shape:", sample_x.shape, "| label index:", sample_y, "->", CLASSES[sample_y])

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# One batch from train loader
batch_x, batch_y = next(iter(train_loader))
print("Train batch: x shape =", batch_x.shape, ", y shape =", batch_y.shape)

## 5. Model: 

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import pennylane as qml

# --- Quantum layer (PennyLane) ---
n_qubits = N_QUBITS
dev = qml.device("default.qubit", wires=n_qubits)
print("Quantum device: default.qubit, wires =", n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def quantum_circuit(inputs):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    for _ in range(2):
        for i in range(n_qubits):
            qml.RY(0.0, wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


def make_quantum_layer(dim_in, n_qubits):
    return nn.Sequential(
        nn.Linear(dim_in, n_qubits),
        nn.Tanh()
    )


class QuantumLayerTorch(nn.Module):
    def __init__(self, dim_in, n_qubits):
        super().__init__()
        self.encoder = make_quantum_layer(dim_in, n_qubits)

    def forward(self, x):
        angles = self.encoder(x) * torch.pi
        out_list = []

        for b in range(angles.shape[0]):
            o = quantum_circuit(angles[b])
            out_list.append(
                torch.stack([
                    torch.as_tensor(oo, device=x.device, dtype=x.dtype)
                    for oo in o
                ])
            )

        return torch.stack(out_list)


class HybridModel(nn.Module):
    def __init__(self, num_classes=14, lstm_hidden=256, reduced_dim=6, n_qubits=4):
        super().__init__()

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False

        feat_dim = 2048
        self.lstm = nn.LSTM(feat_dim, lstm_hidden, batch_first=True)

        self.reducer = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Linear(64, reduced_dim)
        )

        self.quantum = QuantumLayerTorch(reduced_dim, n_qubits)

        self.classifier = nn.Sequential(
            nn.Linear(reduced_dim + n_qubits, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

        print(
            f"HybridModel: backbone=ResNet50 (frozen), "
            f"lstm_hidden={lstm_hidden}, reduced_dim={reduced_dim}, "
            f"n_qubits={n_qubits}, num_classes={num_classes}"
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        with torch.no_grad():
            feats = self.backbone(x).squeeze(-1).squeeze(-1)

        feats = feats.view(B, T, -1)
        out, _ = self.lstm(feats)
        seq_feat = out[:, -1, :]

        reduced = self.reducer(seq_feat)
        quantum_feat = self.quantum(reduced)

        fused = torch.cat([reduced, quantum_feat], dim=1)
        logits = self.classifier(fused)

        return logits

model = HybridModel(NUM_CLASSES, lstm_hidden=256, reduced_dim=REDUCED_DIM, n_qubits=N_QUBITS).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total params:", total, "| Trainable:", trainable)

model.eval()
with torch.no_grad():
    dummy = torch.randn(2, FRAMES_PER_VIDEO, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    logits = model(dummy)

print("Logits shape:", logits.shape)

## 6. Fix quantum circuit: use learnable angles in circuit

In [ ]:
import math

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def quantum_circuit(angles):
    for i in range(min(len(angles), n_qubits)):
        qml.RY(angles[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


class QuantumLayerTorch(nn.Module):
    def __init__(self, dim_in, n_qubits):
        super().__init__()
        self.encoder = nn.Linear(dim_in, n_qubits)

    def forward(self, x):
        angles = self.encoder(x) * math.pi
        out_list = []

        for b in range(angles.shape[0]):
            o = quantum_circuit(angles[b])
            out_list.append(
                torch.stack([
                    torch.as_tensor(oo, device=x.device, dtype=x.dtype)
                    for oo in o
                ])
            )

        return torch.stack(out_list)


## 7. Training loop

In [ ]:
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

def train_epoch(loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(loader, desc="Train"):
        x = x.to(DEVICE, dtype=torch.float32)
        y = y.to(DEVICE)
        optimizer.zero_grad() 
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def eval_epoch(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, dtype=torch.float32)
            y = y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

best_acc = 0.0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(train_loader)
    test_loss, test_acc = eval_epoch(test_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} Test Loss: {test_loss:.4f} Test Acc: {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": CLASSES,
            "epoch": epoch,
            "config": {"IMG_SIZE": IMG_SIZE, "FRAMES_PER_VIDEO": FRAMES_PER_VIDEO, "REDUCED_DIM": REDUCED_DIM, "N_QUBITS": N_QUBITS}
        }, "model.h5")
        print("  -> saved best model.")
print("Best test accuracy:", best_acc)

## 8. Save notebook artifact (Kaggle) or download `model.h5`

After training, use **model.h5** in your Flask backend for inference.